In [0]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, trim, upper, lower,
    when, coalesce, sha2, concat_ws, regexp_replace,
    round as spark_round
)
from pyspark.sql.types import IntegerType, DoubleType, StringType

In [0]:
#1: Configuración de rutas
storage_account_name = "stfrauddatabricks"

container_bronze = "bronze"
container_silver = "silver"

project_folder = "fraud_detection"

bronze_base_path = f"abfss://{container_bronze}@{storage_account_name}.dfs.core.windows.net/{project_folder}"
silver_base_path = f"abfss://{container_silver}@{storage_account_name}.dfs.core.windows.net/{project_folder}"

print("BRONZE path:", bronze_base_path)
print("SILVER path:", silver_base_path)

In [0]:
#2: Rutas Bronze de entrada

bronze_transactions_path = f"{bronze_base_path}/transactions_delta"
bronze_transaction_type_risk_path = f"{bronze_base_path}/transaction_type_risk_delta"
bronze_customer_profile_path = f"{bronze_base_path}/customer_account_profile_delta"

print("Bronze transacciones      :", bronze_transactions_path)
print("Bronze riesgo por tipo    :", bronze_transaction_type_risk_path)
print("Bronze perfil de cuentas  :", bronze_customer_profile_path)

In [0]:
# 3: Rutas Silver de salida
silver_transactions_path = f"{silver_base_path}/transactions_clean_enriched_delta"
silver_transaction_type_risk_path = f"{silver_base_path}/dim_transaction_type_risk_delta"
silver_customer_profile_path = f"{silver_base_path}/dim_customer_account_profile_delta"

print("Silver transacciones enriquecidas :", silver_transactions_path)
print("Silver dim riesgo por tipo        :", silver_transaction_type_risk_path)
print("Silver dim perfil de cuentas      :", silver_customer_profile_path)

In [0]:
#4: Leer datos desde Bronze

df_transactions_bronze = spark.read.format("delta").load(bronze_transactions_path)
df_transaction_type_risk_bronze = spark.read.format("delta").load(bronze_transaction_type_risk_path)
df_customer_profile_bronze = spark.read.format("delta").load(bronze_customer_profile_path)

print("Registros Bronze:")
print("Transacciones      :", df_transactions_bronze.count())
print("Riesgo por tipo    :", df_transaction_type_risk_bronze.count())
print("Perfil de cuentas  :", df_customer_profile_bronze.count())

In [0]:
# Limpiar dimensión de riesgo por tipo
# Revisamos columnas disponibles
print(df_transaction_type_risk_bronze.columns)
display(df_transaction_type_risk_bronze)

In [0]:
# COMMAND ----------

from pyspark.sql.functions import upper, trim, col, when, lit, current_timestamp
from pyspark.sql.types import IntegerType

# Tomamos el dataframe Bronze de riesgo por tipo
df_risk_tmp = df_transaction_type_risk_bronze

# Mostrar columnas originales para validar
print("Columnas originales:")
print(df_risk_tmp.columns)

# Estandarizar nombres de columnas posibles
for old_col, new_col in {
    "type": "transaction_type",
    "Type": "transaction_type",
    "TYPE": "transaction_type",
    "transactionType": "transaction_type",
    "transaction_type": "transaction_type",
    "risk": "risk_level",
    "Risk": "risk_level",
    "riskLevel": "risk_level",
    "risk_level": "risk_level",
    "riskScore": "risk_score",
    "score": "risk_score",
    "risk_score": "risk_score"
}.items():
    if old_col in df_risk_tmp.columns and old_col != new_col:
        df_risk_tmp = df_risk_tmp.withColumnRenamed(old_col, new_col)

print("Columnas después de renombrar:")
print(df_risk_tmp.columns)

# Validación para evitar error si no existe transaction_type
if "transaction_type" not in df_risk_tmp.columns:
    raise Exception(
        "No se encontró una columna de tipo de transacción. "
        "Revisa si tu archivo tiene una columna llamada type, Type, transactionType o transaction_type."
    )

# Quitamos columnas técnicas de Bronze si existen
columnas_excluir = ["source_file", "ingestion_timestamp", "bronze_layer"]
columnas_finales = [c for c in df_risk_tmp.columns if c not in columnas_excluir]

df_transaction_type_risk_silver = (
    df_risk_tmp
    .select(columnas_finales)
    .withColumn("transaction_type", upper(trim(col("transaction_type"))))
)

# Si no existe risk_level, lo creamos como UNKNOWN
if "risk_level" not in df_transaction_type_risk_silver.columns:
    df_transaction_type_risk_silver = df_transaction_type_risk_silver.withColumn(
        "risk_level",
        lit("UNKNOWN")
    )

# Si no existe risk_score, lo creamos según el nivel de riesgo
if "risk_score" not in df_transaction_type_risk_silver.columns:
    df_transaction_type_risk_silver = df_transaction_type_risk_silver.withColumn(
        "risk_score",
        when(upper(col("risk_level")).isin("HIGH", "ALTO"), lit(90))
        .when(upper(col("risk_level")).isin("MEDIUM", "MEDIO"), lit(60))
        .when(upper(col("risk_level")).isin("LOW", "BAJO"), lit(30))
        .otherwise(lit(50))
    )

df_transaction_type_risk_silver = (
    df_transaction_type_risk_silver
    .withColumn("risk_level", upper(trim(col("risk_level"))))
    .withColumn("risk_score", col("risk_score").cast(IntegerType()))
    .dropDuplicates(["transaction_type"])
    .withColumn("silver_updated_at", current_timestamp())
)

display(df_transaction_type_risk_silver)

In [0]:
# COMMAND ----------

from pyspark.sql.functions import col, upper, trim, current_timestamp
from pyspark.sql.types import IntegerType, DoubleType

# Revisamos columnas disponibles del dataset de perfiles
print("Columnas perfil de cuentas:")
print(df_customer_profile_bronze.columns)

display(df_customer_profile_bronze.limit(10))

In [0]:
# COMMAND ----------

df_customer_profile_silver = (
    df_customer_profile_bronze
    .select(
        "account_id",
        "account_type",
        "customer_segment",
        "kyc_status",
        "account_age_days",
        "country",
        "region",
        "preferred_channel",
        "device_type",
        "device_risk_score",
        "login_failed_attempts",
        "avg_monthly_txn_count",
        "avg_monthly_amount",
        "is_high_risk_profile",
        "profile_created_at"
    )
    .withColumn("account_id", trim(col("account_id")))
    .withColumn("account_type", upper(trim(col("account_type"))))
    .withColumn("customer_segment", upper(trim(col("customer_segment"))))
    .withColumn("kyc_status", upper(trim(col("kyc_status"))))
    .withColumn("country", upper(trim(col("country"))))
    .withColumn("region", trim(col("region")))
    .withColumn("preferred_channel", upper(trim(col("preferred_channel"))))
    .withColumn("device_type", upper(trim(col("device_type"))))
    .withColumn("account_age_days", col("account_age_days").cast(IntegerType()))
    .withColumn("device_risk_score", col("device_risk_score").cast(IntegerType()))
    .withColumn("login_failed_attempts", col("login_failed_attempts").cast(IntegerType()))
    .withColumn("avg_monthly_txn_count", col("avg_monthly_txn_count").cast(IntegerType()))
    .withColumn("avg_monthly_amount", col("avg_monthly_amount").cast(DoubleType()))
    .withColumn("is_high_risk_profile", col("is_high_risk_profile").cast(IntegerType()))
    .dropDuplicates(["account_id"])
    .withColumn("silver_updated_at", current_timestamp())
)

display(df_customer_profile_silver.limit(10))

In [0]:
#5. Limpieza de transacciones
# COMMAND ----------

from pyspark.sql.functions import (
    col, upper, trim, sha2, concat_ws, lit, when,
    current_timestamp, coalesce
)
from pyspark.sql.functions import round as spark_round
from pyspark.sql.types import IntegerType, DoubleType, StringType

print("Columnas transacciones Bronze:")
print(df_transactions_bronze.columns)

In [0]:
# COMMAND ----------

df_transactions_silver_base = (
    df_transactions_bronze
    .select(
        "step",
        "type",
        "amount",
        "nameOrig",
        "oldbalanceOrg",
        "newbalanceOrig",
        "nameDest",
        "oldbalanceDest",
        "newbalanceDest",
        "isFraud",
        "isFlaggedFraud"
    )
    .withColumnRenamed("type", "transaction_type")
    .withColumnRenamed("nameOrig", "origin_account_id")
    .withColumnRenamed("nameDest", "destination_account_id")
    .withColumnRenamed("oldbalanceOrg", "origin_old_balance")
    .withColumnRenamed("newbalanceOrig", "origin_new_balance")
    .withColumnRenamed("oldbalanceDest", "destination_old_balance")
    .withColumnRenamed("newbalanceDest", "destination_new_balance")
    .withColumnRenamed("isFraud", "is_fraud")
    .withColumnRenamed("isFlaggedFraud", "is_flagged_fraud")
    .withColumn("step", col("step").cast(IntegerType()))
    .withColumn("transaction_type", upper(trim(col("transaction_type"))))
    .withColumn("amount", spark_round(col("amount").cast(DoubleType()), 2))
    .withColumn("origin_account_id", trim(col("origin_account_id")))
    .withColumn("destination_account_id", trim(col("destination_account_id")))
    .withColumn("origin_old_balance", col("origin_old_balance").cast(DoubleType()))
    .withColumn("origin_new_balance", col("origin_new_balance").cast(DoubleType()))
    .withColumn("destination_old_balance", col("destination_old_balance").cast(DoubleType()))
    .withColumn("destination_new_balance", col("destination_new_balance").cast(DoubleType()))
    .withColumn("is_fraud", col("is_fraud").cast(IntegerType()))
    .withColumn("is_flagged_fraud", col("is_flagged_fraud").cast(IntegerType()))
)

display(df_transactions_silver_base.limit(10))

In [0]:
#6  columnas de calidad y reglas iniciales
# COMMAND ----------

df_transactions_silver_base = (
    df_transactions_silver_base
    .withColumn(
        "transaction_id",
        sha2(
            concat_ws(
                "|",
                col("step").cast(StringType()),
                col("transaction_type"),
                col("origin_account_id"),
                col("destination_account_id"),
                col("amount").cast(StringType()),
                col("origin_old_balance").cast(StringType()),
                col("origin_new_balance").cast(StringType())
            ),
            256
        )
    )
    .withColumn(
        "origin_balance_diff",
        spark_round(col("origin_old_balance") - col("origin_new_balance"), 2)
    )
    .withColumn(
        "destination_balance_diff",
        spark_round(col("destination_new_balance") - col("destination_old_balance"), 2)
    )
    .withColumn(
        "origin_balance_inconsistent",
        when(
            (col("transaction_type").isin("TRANSFER", "CASH_OUT", "PAYMENT", "DEBIT")) &
            (spark_round(col("origin_old_balance") - col("origin_new_balance"), 2) != col("amount")),
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "destination_balance_inconsistent",
        when(
            (col("transaction_type").isin("TRANSFER", "CASH_OUT")) &
            (col("destination_old_balance") > 0) &
            (spark_round(col("destination_new_balance") - col("destination_old_balance"), 2) != col("amount")),
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "amount_category",
        when(col("amount") >= 100000, lit("VERY_HIGH"))
        .when(col("amount") >= 10000, lit("HIGH"))
        .when(col("amount") >= 1000, lit("MEDIUM"))
        .otherwise(lit("LOW"))
    )
    .dropDuplicates(["transaction_id"])
)

display(df_transactions_silver_base.limit(10))

In [0]:
# Transacciones
# COMMAND ----------

df_origin_profile = (
    df_customer_profile_silver
    .select(
        col("account_id").alias("origin_account_id"),
        col("account_type").alias("origin_account_type"),
        col("customer_segment").alias("origin_customer_segment"),
        col("kyc_status").alias("origin_kyc_status"),
        col("account_age_days").alias("origin_account_age_days"),
        col("country").alias("origin_country"),
        col("region").alias("origin_region"),
        col("preferred_channel").alias("origin_preferred_channel"),
        col("device_type").alias("origin_device_type"),
        col("device_risk_score").alias("origin_device_risk_score"),
        col("login_failed_attempts").alias("origin_login_failed_attempts"),
        col("is_high_risk_profile").alias("origin_is_high_risk_profile")
    )
)

df_destination_profile = (
    df_customer_profile_silver
    .select(
        col("account_id").alias("destination_account_id"),
        col("account_type").alias("destination_account_type"),
        col("customer_segment").alias("destination_customer_segment"),
        col("kyc_status").alias("destination_kyc_status"),
        col("account_age_days").alias("destination_account_age_days"),
        col("country").alias("destination_country"),
        col("region").alias("destination_region"),
        col("preferred_channel").alias("destination_preferred_channel"),
        col("device_type").alias("destination_device_type"),
        col("device_risk_score").alias("destination_device_risk_score"),
        col("login_failed_attempts").alias("destination_login_failed_attempts"),
        col("is_high_risk_profile").alias("destination_is_high_risk_profile")
    )
)

In [0]:
# COMMAND ----------

df_transactions_silver_enriched = (
    df_transactions_silver_base
    .join(
        df_transaction_type_risk_silver,
        ["transaction_type"],
        "left"
    )
    .join(
        df_origin_profile,
        ["origin_account_id"],
        "left"
    )
    .join(
        df_destination_profile,
        ["destination_account_id"],
        "left"
    )
)

display(df_transactions_silver_enriched.limit(10))

display(df_transactions_silver_enriched.limit(10))

In [0]:
# COMMAND ----------

from pyspark.sql.functions import coalesce, col, lit, when, current_timestamp

df_transactions_silver_enriched = (
    df_transactions_silver_enriched
    .withColumn("risk_level", coalesce(col("risk_level"), lit("UNKNOWN")))
    .withColumn("risk_score", coalesce(col("risk_score"), lit(0)))
    
    .withColumn("origin_account_type", coalesce(col("origin_account_type"), lit("UNKNOWN")))
    .withColumn("origin_customer_segment", coalesce(col("origin_customer_segment"), lit("UNKNOWN")))
    .withColumn("origin_kyc_status", coalesce(col("origin_kyc_status"), lit("UNKNOWN")))
    .withColumn("origin_country", coalesce(col("origin_country"), lit("UNKNOWN")))
    .withColumn("origin_region", coalesce(col("origin_region"), lit("UNKNOWN")))
    .withColumn("origin_preferred_channel", coalesce(col("origin_preferred_channel"), lit("UNKNOWN")))
    .withColumn("origin_device_type", coalesce(col("origin_device_type"), lit("UNKNOWN")))
    .withColumn("origin_device_risk_score", coalesce(col("origin_device_risk_score"), lit(0)))
    .withColumn("origin_login_failed_attempts", coalesce(col("origin_login_failed_attempts"), lit(0)))
    .withColumn("origin_is_high_risk_profile", coalesce(col("origin_is_high_risk_profile"), lit(0)))
    
    .withColumn("destination_account_type", coalesce(col("destination_account_type"), lit("UNKNOWN")))
    .withColumn("destination_customer_segment", coalesce(col("destination_customer_segment"), lit("UNKNOWN")))
    .withColumn("destination_kyc_status", coalesce(col("destination_kyc_status"), lit("UNKNOWN")))
    .withColumn("destination_country", coalesce(col("destination_country"), lit("UNKNOWN")))
    .withColumn("destination_region", coalesce(col("destination_region"), lit("UNKNOWN")))
    .withColumn("destination_preferred_channel", coalesce(col("destination_preferred_channel"), lit("UNKNOWN")))
    .withColumn("destination_device_type", coalesce(col("destination_device_type"), lit("UNKNOWN")))
    .withColumn("destination_device_risk_score", coalesce(col("destination_device_risk_score"), lit(0)))
    .withColumn("destination_login_failed_attempts", coalesce(col("destination_login_failed_attempts"), lit(0)))
    .withColumn("destination_is_high_risk_profile", coalesce(col("destination_is_high_risk_profile"), lit(0)))
    
    .withColumn(
        "profile_match_flag",
        when(
            (col("origin_account_type") != "UNKNOWN") |
            (col("destination_account_type") != "UNKNOWN"),
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "silver_fraud_risk_score",
        (
            col("risk_score") * 10 +
            col("origin_device_risk_score") +
            col("destination_device_risk_score") +
            (col("origin_is_high_risk_profile") * 20) +
            (col("destination_is_high_risk_profile") * 20) +
            (col("origin_balance_inconsistent") * 10) +
            (col("destination_balance_inconsistent") * 10)
        )
    )
    .withColumn(
        "silver_fraud_risk_level",
        when(col("silver_fraud_risk_score") >= 140, lit("CRITICAL"))
        .when(col("silver_fraud_risk_score") >= 100, lit("HIGH"))
        .when(col("silver_fraud_risk_score") >= 70, lit("MEDIUM"))
        .otherwise(lit("LOW"))
    )
    .withColumn("silver_updated_at", current_timestamp())
)

display(df_transactions_silver_enriched.limit(10))

In [0]:
# COMMAND ----------

from collections import Counter

column_counts = Counter(df_transactions_silver_enriched.columns)
duplicate_columns = [column for column, count in column_counts.items() if count > 1]

print("Columnas duplicadas:", duplicate_columns)
print("Cantidad de columnas:", len(df_transactions_silver_enriched.columns))

In [0]:
#Validaciones
# COMMAND ----------

print("Validaciones Silver:")

print("Transacciones limpias y enriquecidas :", df_transactions_silver_enriched.count())
print("Dim riesgo por tipo                  :", df_transaction_type_risk_silver.count())
print("Dim perfil de cuentas                :", df_customer_profile_silver.count())

In [0]:
print("Distribución de fraude real:")
df_transactions_silver_enriched.groupBy("is_fraud").count().show()

print("Distribución de nivel de riesgo Silver:")
df_transactions_silver_enriched.groupBy("silver_fraud_risk_level").count().show()

print("Coincidencia con perfiles:")
df_transactions_silver_enriched.groupBy("profile_match_flag").count().show()

print("Distribución por tipo de transacción:")

In [0]:
# Guardar y validar Silver
# COMMAND ----------

silver_transactions_path = f"{silver_base_path}/transactions_clean_enriched_delta"
silver_transaction_type_risk_path = f"{silver_base_path}/dim_transaction_type_risk_delta"
silver_customer_profile_path = f"{silver_base_path}/dim_customer_account_profile_delta"

print("Silver transacciones enriquecidas :", silver_transactions_path)
print("Silver dim riesgo por tipo        :", silver_transaction_type_risk_path)
print("Silver dim perfil de cuentas      :", silver_customer_profile_path)

In [0]:
# COMMAND ----------

(
    df_transactions_silver_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_transactions_path)
)

(
    df_transaction_type_risk_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_transaction_type_risk_path)
)

(
    df_customer_profile_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_customer_profile_path)
)

print("Datos guardados correctamente en la capa Silver.")

In [0]:
# COMMAND ----------

df_transactions_silver_check = spark.read.format("delta").load(silver_transactions_path)
df_transaction_type_risk_silver_check = spark.read.format("delta").load(silver_transaction_type_risk_path)
df_customer_profile_silver_check = spark.read.format("delta").load(silver_customer_profile_path)

print("Validación final Silver:")
print("Transacciones Silver :", df_transactions_silver_check.count())
print("Riesgo tipo Silver   :", df_transaction_type_risk_silver_check.count())
print("Perfil Silver        :", df_customer_profile_silver_check.count())

In [0]:
# COMMAND ----------

df_transactions_silver_check.printSchema()